# Hyperparameter Optimization - Theory Notebook

This notebook is the interactive companion to `notes.md`.
It treats hyperparameter optimization as noisy, budgeted outer-loop optimization over configurations $\boldsymbol{\lambda}$.

The code uses synthetic objectives so every cell can run without external data.

In [ ]:
import math
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style="whitegrid", palette="colorblind")
        HAS_SNS = True
    except ImportError:
        HAS_SNS = False
    mpl.rcParams.update({
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "legend.framealpha": 0.85,
        "lines.linewidth": 2.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.bbox": "tight",
        "savefig.dpi": 150,
    })
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    HAS_SNS = False

COLORS = {
    "primary": "#0077BB",
    "secondary": "#EE7733",
    "tertiary": "#009988",
    "error": "#CC3311",
    "neutral": "#555555",
    "highlight": "#EE3377",
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

def check_true(name, condition):
    ok = bool(condition)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    assert ok, name

def check_close(name, value, target, tol=1e-8):
    ok = abs(float(value) - float(target)) <= tol
    print(f"{'PASS' if ok else 'FAIL'} - {name}: {value} vs {target}")
    assert ok, name

def normal_pdf(z):
    z = np.asarray(z, dtype=float)
    return np.exp(-0.5 * z * z) / math.sqrt(2.0 * math.pi)

def normal_cdf(z):
    z = np.asarray(z, dtype=float)
    erf_vec = np.vectorize(math.erf)
    return 0.5 * (1.0 + erf_vec(z / math.sqrt(2.0)))

def synthetic_hpo_objective(lr, weight_decay, rank=8, noise=0.0, seed=None):
    rng = np.random.default_rng(seed)
    x = math.log10(lr)
    wd = math.log10(weight_decay + 1e-8)
    rank_penalty = 0.002 * (rank - 16) ** 2 / 16.0
    bowl = (x + 3.7) ** 2 + 0.18 * (wd + 2.2) ** 2 + rank_penalty
    interaction = 0.08 * math.sin(4.0 * x + 1.5 * wd)
    return 0.35 + bowl + interaction + rng.normal(0.0, noise)

def sample_log_uniform(low_exp, high_exp, n, rng):
    return 10 ** rng.uniform(low_exp, high_exp, size=n)

def rbf_kernel(X1, X2, lengthscale=0.25, variance=1.0):
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    sq = np.sum((X1[:, None, :] - X2[None, :, :]) ** 2, axis=2)
    return variance * np.exp(-0.5 * sq / (lengthscale ** 2))

def gp_predict(X_train, y_train, X_test, lengthscale=0.25, noise=1e-6):
    X_train = np.atleast_2d(X_train)
    X_test = np.atleast_2d(X_test)
    y_train = np.asarray(y_train)
    K = rbf_kernel(X_train, X_train, lengthscale) + noise * np.eye(len(X_train))
    Ks = rbf_kernel(X_train, X_test, lengthscale)
    Kss = rbf_kernel(X_test, X_test, lengthscale)
    L = la.cholesky(K)
    alpha = la.solve(L.T, la.solve(L, y_train))
    mu = Ks.T @ alpha
    v = la.solve(L, Ks)
    cov = Kss - v.T @ v
    var = np.maximum(np.diag(cov), 1e-12)
    return mu, var

def expected_improvement(mu, sigma, best, xi=0.0):
    sigma = np.maximum(np.asarray(sigma), 1e-12)
    z = (best - xi - np.asarray(mu)) / sigma
    return (best - xi - np.asarray(mu)) * normal_cdf(z) + sigma * normal_pdf(z)

print("Setup complete.")
print("NumPy version:", np.__version__)
print("Matplotlib available:", HAS_MPL)

---

## 1.1 Hyperparameters as the outer loop

We model training as an expensive function from configuration to validation score.

In [ ]:
header("1.1 Synthetic outer-loop objective")
cfg = {"lr": 2e-4, "weight_decay": 1e-2, "rank": 16}
score = synthetic_hpo_objective(**cfg, noise=0.0)
print("configuration:", cfg)
print(f"validation loss proxy: {score:.6f}")
check_true("score is finite", np.isfinite(score))

## 1.2 The objective is noisy

Repeated trials of the same configuration can produce different validation scores.

In [ ]:
header("1.2 Repeated noisy evaluations")
scores = [synthetic_hpo_objective(2e-4, 1e-2, 16, noise=0.015, seed=s) for s in range(20)]
print("mean score:", np.mean(scores))
print("std score:", np.std(scores, ddof=1))
check_true("noise creates nonzero variation", np.std(scores, ddof=1) > 0.005)

## 1.3 Budget arithmetic

Total cost is the sum of trial costs, not just the number of configurations.

In [ ]:
header("1.3 Budget arithmetic")
trials = 60
minutes_per_trial = 35
gpu_hours = trials * minutes_per_trial / 60
print(f"{trials} trials x {minutes_per_trial} minutes = {gpu_hours:.2f} GPU-hours")
check_close("budget computation", gpu_hours, 35.0)

## 1.4 Low-fidelity signals

Multi-fidelity methods need early scores to correlate with final scores.

In [ ]:
header("1.4 Early/final correlation")
rng = np.random.default_rng(7)
true_quality = rng.normal(0, 1, size=80)
early = true_quality + rng.normal(0, 0.8, size=80)
final = true_quality + rng.normal(0, 0.25, size=80)
corr = np.corrcoef(early, final)[0, 1]
print(f"early/final rank signal correlation: {corr:.3f}")
check_true("early signal is informative", corr > 0.5)

## 1.5 Algorithm families

Different HPO methods use different information.

In [ ]:
header("1.5 Method summary")
rows = [
    ("grid", "no past scores", "no resource allocation"),
    ("random", "no past scores", "parallel baseline"),
    ("Bayesian optimization", "surrogate model", "sample efficient"),
    ("Hyperband/ASHA", "intermediate scores", "multi-fidelity"),
    ("PBT", "population scores", "dynamic hyperparameters"),
]
for name, signal, strength in rows:
    print(f"{name:24s} | {signal:20s} | {strength}")
check_true("five families listed", len(rows) == 5)

---

## 2.1 Configuration spaces

A configuration combines numeric, categorical, and conditional choices.

In [ ]:
header("2.1 Configuration dictionary")
config = {
    "optimizer": "AdamW",
    "lr": 3e-4,
    "weight_decay": 0.01,
    "lora_rank": 16,
    "batch_size": 64,
}
for key, value in config.items():
    print(f"{key:14s}: {value}")
check_true("learning rate is positive", config["lr"] > 0)

## 2.2 Log-uniform sampling

Learning rates are usually searched on a logarithmic scale.

In [ ]:
header("2.2 Log-uniform samples")
rng = np.random.default_rng(42)
lrs = sample_log_uniform(-5, -2, 8, rng)
print("learning-rate samples:", lrs)
print("log10 samples:", np.log10(lrs))
check_true("all samples in range", np.all((lrs >= 1e-5) & (lrs <= 1e-2)))

## 2.3 Conditional spaces

Some hyperparameters are active only under a branch.

In [ ]:
header("2.3 Conditional optimizer branch")
rng = np.random.default_rng(3)
optimizer = rng.choice(["AdamW", "SGD", "Adafactor"])
trial = {"optimizer": optimizer}
if optimizer == "AdamW":
    trial["beta2"] = 1.0 - 10 ** rng.uniform(-4, -1)
elif optimizer == "SGD":
    trial["momentum"] = rng.uniform(0.0, 0.99)
else:
    trial["relative_step"] = bool(rng.integers(0, 2))
print(trial)
check_true("branch-specific field added", len(trial) == 2)

## 2.4 Validation standard error

Finite validation data creates metric uncertainty.

In [ ]:
header("2.4 Accuracy standard error")
p_hat = 0.84
n_val = 1500
se = math.sqrt(p_hat * (1 - p_hat) / n_val)
print(f"accuracy estimate: {p_hat:.3f} +/- {1.96 * se:.3f} approximately")
check_true("standard error is small but nonzero", 0 < se < 0.02)

## 2.5 Pareto dominance

Multi-objective HPO often returns a frontier rather than one winner.

In [ ]:
header("2.5 Pareto frontier helper")
points = np.array([
    [0.42, 120],
    [0.40, 180],
    [0.45, 90],
    [0.41, 110],
    [0.39, 240],
], dtype=float)

def is_dominated(i, pts):
    return any(np.all(pts[j] <= pts[i]) and np.any(pts[j] < pts[i]) for j in range(len(pts)) if j != i)

frontier = [i for i in range(len(points)) if not is_dominated(i, points)]
print("frontier indices:", frontier)
print("frontier points:", points[frontier])
check_true("configuration A is dominated by D", is_dominated(0, points))

---

## 3.1 Manual search as range discovery

In [ ]:
header("3.1 Manual pilot range")
pilot_lrs = np.array([1e-5, 3e-5, 1e-4, 3e-4, 1e-3])
losses = np.array([synthetic_hpo_objective(lr, 1e-2, 16, seed=11) for lr in pilot_lrs])
best_idx = int(np.argmin(losses))
print("pilot learning rates:", pilot_lrs)
print("pilot losses:", losses)
print("best pilot lr:", pilot_lrs[best_idx])
check_true("best index valid", 0 <= best_idx < len(pilot_lrs))

## 3.2 Grid size grows multiplicatively

In [ ]:
header("3.2 Grid size")
choices = [4, 3, 4, 2]
size = int(np.prod(choices))
print("choices per dimension:", choices)
print("total grid trials:", size)
check_close("grid size", size, 96)

## 3.3 Grid search wastes resolution on irrelevant dimensions

In [ ]:
header("3.3 Effective dimension")
grid_values_per_dim = 3
total_dims = 10
important_dims = 2
total_points = grid_values_per_dim ** total_dims
important_resolution = grid_values_per_dim ** important_dims
print("total grid points:", total_points)
print("distinct important settings:", important_resolution)
check_true("total points dwarf important settings", total_points > 1000 * important_resolution)

## 3.4 Random search hit probability

In [ ]:
header("3.4 Hit probability")
q_good = 0.04
for n in [10, 25, 50, 100]:
    p_hit = 1 - (1 - q_good) ** n
    print(f"N={n:3d}: P(hit)={p_hit:.3f}")
check_true("100 trials likely hit", 1 - (1 - q_good) ** 100 > 0.98)

## 3.5 Random search implementation

In [ ]:
header("3.5 Random search on synthetic objective")
rng = np.random.default_rng(123)
trials = []
for j in range(80):
    lr = float(sample_log_uniform(-5, -2, 1, rng)[0])
    wd = float(sample_log_uniform(-6, -1, 1, rng)[0])
    rank = int(rng.choice([4, 8, 16, 32]))
    y = synthetic_hpo_objective(lr, wd, rank, noise=0.01, seed=j)
    trials.append((y, lr, wd, rank))
best = min(trials, key=lambda row: row[0])
print("best random-search trial:", best)
check_true("best score is finite", np.isfinite(best[0]))

## 3.6 Latin hypercube coverage

In [ ]:
header("3.6 Latin hypercube in two dimensions")
rng = np.random.default_rng(9)
n = 10
lhs = np.zeros((n, 2))
for d in range(2):
    perm = rng.permutation(n)
    lhs[:, d] = (perm + rng.uniform(size=n)) / n
print(np.round(lhs, 3))
check_true("all coordinates are within unit square", np.all((lhs >= 0) & (lhs <= 1)))

---

## 4.1 Gaussian-process prediction

In [ ]:
header("4.1 GP posterior on one-dimensional objective")
rng = np.random.default_rng(4)
X_train = np.array([[-0.9], [-0.4], [0.1], [0.7]])
y_train = (X_train[:, 0] - 0.2) ** 2 + rng.normal(0, 0.01, size=len(X_train))
X_grid = np.linspace(-1, 1, 101)[:, None]
mu, var = gp_predict(X_train, y_train, X_grid, lengthscale=0.35, noise=1e-5)
print("posterior mean min location:", float(X_grid[np.argmin(mu), 0]))
print("average posterior std:", float(np.mean(np.sqrt(var))))
check_true("variance is nonnegative", np.all(var >= 0))

## 4.2 Probability and expected improvement

In [ ]:
header("4.2 Acquisition values")
best = float(np.min(y_train))
sigma = np.sqrt(var)
ei = expected_improvement(mu, sigma, best)
pi = normal_cdf((best - mu) / np.maximum(sigma, 1e-12))
print("best observed:", best)
print("max PI:", float(np.max(pi)))
print("max EI:", float(np.max(ei)))
check_true("expected improvement is nonnegative", np.all(ei >= -1e-12))

## 4.3 One Bayesian-optimization step

In [ ]:
header("4.3 Select next candidate by EI")
next_idx = int(np.argmax(ei))
next_x = float(X_grid[next_idx, 0])
print("next candidate:", next_x)
print("mu, std, EI:", float(mu[next_idx]), float(sigma[next_idx]), float(ei[next_idx]))
check_true("candidate lies in domain", -1 <= next_x <= 1)

## 4.4 Mini Bayesian-optimization loop

In [ ]:
header("4.4 Mini BO loop")
rng = np.random.default_rng(5)
X_obs = rng.uniform(-1, 1, size=(4, 1))
y_obs = (X_obs[:, 0] - 0.25) ** 2 + 0.05 * np.sin(12 * X_obs[:, 0])
grid = np.linspace(-1, 1, 301)[:, None]
for step in range(8):
    mu_step, var_step = gp_predict(X_obs, y_obs, grid, lengthscale=0.25, noise=1e-5)
    ei_step = expected_improvement(mu_step, np.sqrt(var_step), np.min(y_obs))
    x_new = grid[int(np.argmax(ei_step))]
    y_new = (x_new[0] - 0.25) ** 2 + 0.05 * np.sin(12 * x_new[0])
    X_obs = np.vstack([X_obs, x_new.reshape(1, 1)])
    y_obs = np.r_[y_obs, y_new]
print("best x after BO loop:", float(X_obs[np.argmin(y_obs), 0]))
print("best y after BO loop:", float(np.min(y_obs)))
check_true("BO found a low value", np.min(y_obs) < 0.03)

## 4.5 TPE-style good/bad density ratio

In [ ]:
header("4.5 TPE-style split")
ys = np.array([row[0] for row in trials])
log_lrs = np.log10([row[1] for row in trials])
cutoff = np.quantile(ys, 0.25)
good = log_lrs[ys <= cutoff]
bad = log_lrs[ys > cutoff]
print("mean log10(lr) for good trials:", np.mean(good))
print("mean log10(lr) for bad trials:", np.mean(bad))
check_true("good and bad groups are both populated", len(good) > 0 and len(bad) > 0)

---

## 5.1 Successive halving

In [ ]:
header("5.1 Successive halving simulation")
rng = np.random.default_rng(12)
qualities = rng.normal(0, 1, size=27)
alive = np.arange(27)
resource = 1
history = []
while len(alive) > 1:
    observed = qualities[alive] + rng.normal(0, 1 / math.sqrt(resource), size=len(alive))
    keep_count = max(1, len(alive) // 3)
    keep = alive[np.argsort(observed)[:keep_count]]
    history.append((resource, len(alive), keep_count))
    alive = keep
    resource *= 3
print("rung history (resource, candidates, kept):", history)
print("winner:", int(alive[0]))
check_true("one winner remains", len(alive) == 1)

## 5.2 Hyperband bracket arithmetic

In [ ]:
header("5.2 Hyperband brackets")
R = 81
eta = 3
s_max = int(math.floor(math.log(R, eta)))
brackets = []
for s in range(s_max, -1, -1):
    n = math.ceil((s_max + 1) / (s + 1) * eta ** s)
    r = R * eta ** (-s)
    brackets.append((s, n, r))
print("s_max:", s_max)
print("brackets (s, n, r):", brackets)
check_true("aggressive bracket starts with most configs", brackets[0][1] > brackets[-1][1])

## 5.3 ASHA-style asynchronous decisions

In [ ]:
header("5.3 ASHA-style percentile cutoff")
rng = np.random.default_rng(17)
rung_scores = rng.normal(0.5, 0.08, size=20)
candidate_score = 0.43
cutoff = np.quantile(rung_scores, 1/3)
decision = "promote" if candidate_score <= cutoff else "stop"
print("rung cutoff:", cutoff)
print("candidate score:", candidate_score)
print("decision:", decision)
check_true("good low score is promoted", decision == "promote")

## 5.4 BOHB intuition

In [ ]:
header("5.4 BOHB-like weighted resampling")
ys = np.array([row[0] for row in trials])
weights = np.exp(-(ys - ys.min()) / 0.05)
weights = weights / weights.sum()
selected = np.random.default_rng(21).choice(len(trials), size=5, replace=False, p=weights)
print("selected historical trial indices:", selected)
print("selected scores:", ys[selected])
print("overall mean score:", ys.mean())
print("selected mean score:", ys[selected].mean())
check_true("weighted selection favors better scores", ys[selected].mean() < ys.mean())

---

## 6.1 Evolutionary mutation

In [ ]:
header("6.1 Mutating learning rates")
rng = np.random.default_rng(31)
parent_lr = 2e-4
children = parent_lr * np.exp(rng.normal(0, 0.5, size=8))
print("parent lr:", parent_lr)
print("children:", children)
check_true("all mutated learning rates are positive", np.all(children > 0))

## 6.2 Population-based training exploit/explore

In [ ]:
header("6.2 PBT exploit/explore")
population = [
    {"score": 0.48, "lr": 1e-4},
    {"score": 0.41, "lr": 2e-4},
    {"score": 0.55, "lr": 8e-5},
    {"score": 0.39, "lr": 3e-4},
]
best = min(population, key=lambda p: p["score"])
worst = max(population, key=lambda p: p["score"])
mutated_lr = best["lr"] * 1.2
print("best member:", best)
print("worst before replacement:", worst)
print("replacement lr:", mutated_lr)
check_true("replacement copies strong member then mutates", mutated_lr > best["lr"])

## 6.3 Schedule parameterization

In [ ]:
header("6.3 Warmup-cosine schedule parameters")
steps = np.arange(0, 11)
peak = 3e-4
warmup = 3
total = 10
lr = np.where(
    steps <= warmup,
    peak * steps / warmup,
    peak * 0.5 * (1 + np.cos(np.pi * (steps - warmup) / (total - warmup))),
)
print("steps:", steps)
print("schedule:", lr)
check_close("peak at warmup", lr[warmup], peak)

## 6.4 Transfer warm start

In [ ]:
header("6.4 Warm-starting from historical studies")
historical_best_lrs = np.array([1.5e-4, 2e-4, 2.5e-4, 3e-4])
center = np.median(np.log10(historical_best_lrs))
proposed_range = (10 ** (center - 0.7), 10 ** (center + 0.7))
print("historical best lrs:", historical_best_lrs)
print("proposed log-centered range:", proposed_range)
check_true("range contains median", proposed_range[0] < np.median(historical_best_lrs) < proposed_range[1])

---

## 7.1 Test-set leakage simulation

In [ ]:
header("7.1 Leakage through repeated test selection")
rng = np.random.default_rng(44)
true_test = 0.50
noisy_test_scores = true_test + rng.normal(0, 0.02, size=200)
selected = noisy_test_scores.min()
print("true mean test score:", true_test)
print("best observed after peeking:", selected)
print("optimism:", true_test - selected)
check_true("peeking creates optimistic score", selected < true_test)

## 7.2 Nested validation concept with polynomial regression

In [ ]:
header("7.2 Simple inner/outer selection")
rng = np.random.default_rng(55)
x = np.linspace(-1, 1, 60)
y = np.sin(3 * x) + rng.normal(0, 0.15, size=len(x))

def fit_poly(x_train, y_train, degree):
    X = np.vander(x_train, degree + 1, increasing=True)
    return la.lstsq(X, y_train, rcond=None)[0]

def mse_poly(coef, x_eval, y_eval):
    X = np.vander(x_eval, len(coef), increasing=True)
    return float(np.mean((X @ coef - y_eval) ** 2))

outer_test = np.arange(0, len(x), 5)
inner_train = np.setdiff1d(np.arange(len(x)), outer_test)
val = inner_train[::4]
train = np.setdiff1d(inner_train, val)
candidates = [1, 2, 3, 5, 9]
val_scores = []
for degree in candidates:
    coef = fit_poly(x[train], y[train], degree)
    val_scores.append(mse_poly(coef, x[val], y[val]))
best_degree = candidates[int(np.argmin(val_scores))]
coef = fit_poly(x[inner_train], y[inner_train], best_degree)
outer_mse = mse_poly(coef, x[outer_test], y[outer_test])
print("validation scores:", dict(zip(candidates, np.round(val_scores, 4))))
print("selected degree:", best_degree)
print("outer MSE:", outer_mse)
check_true("selected degree is in candidate set", best_degree in candidates)

## 7.3 Repeated seeds for final candidates

In [ ]:
header("7.3 Repeat top candidates")
rng = np.random.default_rng(66)
candidate_a = 0.40 + rng.normal(0, 0.015, size=5)
candidate_b = 0.395 + rng.normal(0, 0.03, size=5)
for name, vals in [("A", candidate_a), ("B", candidate_b)]:
    print(name, "mean", vals.mean(), "std", vals.std(ddof=1))
winner = "A" if candidate_a.mean() < candidate_b.mean() else "B"
print("winner by repeated mean:", winner)
check_true("means are finite", np.isfinite(candidate_a.mean()) and np.isfinite(candidate_b.mean()))

## 7.4 Confidence interval for final score

In [ ]:
header("7.4 Confidence interval")
vals = candidate_a
mean = vals.mean()
se = vals.std(ddof=1) / math.sqrt(len(vals))
ci = (mean - 1.96 * se, mean + 1.96 * se)
print("mean:", mean)
print("approx 95% CI:", ci)
check_true("CI has positive width", ci[1] > ci[0])

---

## 8.1 Classical ML search space

In [ ]:
header("8.1 Classical pipeline search")
svm_space = {
    "C": "log-uniform(1e-3, 1e3)",
    "gamma": "log-uniform(1e-5, 1e0)",
    "kernel": ["rbf", "poly"],
}
for k, v in svm_space.items():
    print(k, "->", v)
check_true("C is included", "C" in svm_space)

## 8.2 Deep-learning coupled knobs

In [ ]:
header("8.2 Batch-size and learning-rate coupling")
base_lr = 1e-4
base_batch = 64
batches = np.array([64, 128, 256, 512])
scaled_lrs = base_lr * np.sqrt(batches / base_batch)
print("batches:", batches)
print("sqrt-scaled learning rates:", scaled_lrs)
check_close("base point unchanged", scaled_lrs[0], base_lr)

## 8.3 LLM fine-tuning budget

In [ ]:
header("8.3 LoRA fine-tuning budget")
ranks = [4, 8, 16, 32]
lrs = 12
wd = 3
total = len(ranks) * lrs * wd
budget = 60
print("full factorial trials:", total)
print("available random-search budget:", budget)
print("fraction explored:", budget / total)
check_true("random budget is partial", budget < total)

## 8.4 HPO report checklist

In [ ]:
header("8.4 Minimal reproducibility checklist")
checklist = [
    "search_space",
    "sampler",
    "scheduler",
    "metric",
    "budget",
    "seeds",
    "data_version",
    "code_version",
    "failed_trials",
]
for item in checklist:
    print("[ ]", item)
check_true("checklist has enough fields", len(checklist) >= 8)

## 8.5 Final synthesis

In [ ]:
header("8.5 Strategy selector")
def choose_strategy(trial_cost, has_learning_curves, dimension, workers):
    if has_learning_curves and workers >= 4:
        return "ASHA/Hyperband"
    if trial_cost == "expensive" and dimension <= 10:
        return "Bayesian optimization"
    if dimension >= 10:
        return "random search baseline"
    return "grid or random search"

cases = [
    ("cheap", False, 4, 1),
    ("expensive", False, 5, 1),
    ("expensive", True, 12, 16),
]
for case in cases:
    print(case, "->", choose_strategy(*case))
check_true("selector returns a string", isinstance(choose_strategy(*cases[0]), str))